# Data Cleaning and Preprocessing

**Audit-first workflow:** inventory → standardize → type-fix → validate → missing-value strategy → deduplicate → verify → export.

In [ ]:
from pathlib import Path
import pandas as pd, numpy as np
ROOT=Path.cwd()
while not (ROOT/'data').exists() and ROOT.parent!=ROOT: ROOT=ROOT.parent
df=pd.read_csv(ROOT/'data/raw/titanic_messy.csv',keep_default_na=False,na_values=['','NA','N/A','null','NULL','None','none','-'])
df.head()


## 1. Inventory before fixing anything

In [ ]:
print('Shape:',df.shape)
display(df.isna().sum().sort_values(ascending=False).to_frame('missing'))
print('Exact duplicate rows:',df.duplicated().sum())
print('Duplicate PassengerId rows:',df['PassengerId'].duplicated(keep=False).sum())
display(df.dtypes.to_frame('dtype'))


## 2. Standardize text/categorical values

In [ ]:
for c in df.select_dtypes(include='object').columns:
    df[c]=df[c].map(lambda x:x.strip() if isinstance(x,str) else x)
df['Sex']=df['Sex'].astype('string').str.strip().str.lower().map({'male':'male','m':'male','female':'female','f':'female'})
df['Embarked']=df['Embarked'].astype('string').str.strip().str.lower().map({'s':'S','southampton':'S','c':'C','cherbourg':'C','q':'Q','queenstown':'Q'})
display(df[['Sex','Embarked']].value_counts().head(10))


## 3. Correct data types and validate numeric fields

In [ ]:
df['Pclass']=pd.to_numeric(df['Pclass'].astype(str).str.extract(r'(\d+)',expand=False),errors='coerce')
df['Survived']=pd.to_numeric(df['Survived'].astype(str).str.lower().replace({'yes':1,'no':0}),errors='coerce')
for c in ['PassengerId','Age','SibSp','Parch','Fare']:
    df[c]=pd.to_numeric(df[c].astype(str).str.replace(r'[^0-9.\-]','',regex=True),errors='coerce')
bad_age=df['Age'].notna() & ~df['Age'].between(0,100)
bad_fare=df['Fare'].notna() & (df['Fare']<0)
df.loc[bad_age,'Age']=np.nan
df.loc[bad_fare,'Fare']=np.nan
print('Invalid ages:',int(bad_age.sum()),'Invalid fares:',int(bad_fare.sum()))


## 4. Missing-value handling

In [ ]:
df['Name']=df['Name'].astype(str).str.replace(r'\s+',' ',regex=True).str.strip()
df['Ticket']=df['Ticket'].astype(str).str.replace(r'\s+',' ',regex=True).str.strip().str.upper()
df=df.dropna(subset=['PassengerId']).copy()
df['Embarked']=df['Embarked'].fillna(df['Embarked'].mode().iloc[0])
df['Cabin']=df['Cabin'].fillna('Unknown')
df['Age']=df['Age'].fillna(df.groupby(['Pclass','Sex'])['Age'].transform('median')).fillna(df['Age'].median())
df['Fare']=df['Fare'].fillna(df.groupby('Pclass')['Fare'].transform('median')).fillna(df['Fare'].median())
display(df.isna().sum().sort_values(ascending=False))


## 5. Duplicate handling

In [ ]:
df=df.drop_duplicates().copy()
if df['PassengerId'].duplicated(keep=False).any():
    df['_completeness']=df.notna().sum(axis=1)
    df=(df.sort_values(['PassengerId','_completeness'],ascending=[True,False])
          .drop_duplicates('PassengerId',keep='first')
          .drop(columns='_completeness'))
print('Exact duplicates:',df.duplicated().sum())
print('PassengerId unique:',df['PassengerId'].is_unique)


## 6. Final validation

In [ ]:
expected=['PassengerId','Survived','Pclass','Name','Sex','Age','SibSp','Parch','Ticket','Fare','Cabin','Embarked']
df['PassengerId']=df['PassengerId'].astype('int64')
df['Survived']=df['Survived'].astype('int64')
df['Pclass']=df['Pclass'].astype('int64')
df['SibSp']=df['SibSp'].astype('int64')
df['Parch']=df['Parch'].astype('int64')
df['Age']=df['Age'].astype('float64')
df['Fare']=df['Fare'].astype('float64')
df['Sex']=df['Sex'].astype('category')
df['Embarked']=df['Embarked'].astype('category')
df=df[expected].sort_values('PassengerId').reset_index(drop=True)
assert df.duplicated().sum()==0
assert df['PassengerId'].is_unique
assert df['PassengerId'].notna().all()
assert set(df['Survived'].unique()).issubset({0,1})
assert set(df['Pclass'].unique()).issubset({1,2,3})
assert set(df['Sex'].astype(str).unique()).issubset({'male','female'})
assert set(df['Embarked'].astype(str).unique()).issubset({'S','C','Q'})
assert df['Age'].between(0,100).all()
assert (df['Fare']>=0).all()
print('All validation checks passed.')


## 7. Export

In [ ]:
df.to_csv(ROOT/'data/processed/titanic_cleaned.csv',index=False)
df.to_excel(ROOT/'data/processed/titanic_cleaned.xlsx',index=False)
print('CSV and XLSX exported.')
